# Lab 8: copy a big model's thinking into a small one

*Teach a Small AI to Think* (unicodesnwrites.tech), module 8.

In this notebook you take **Qwen2.5-0.5B-Instruct**, a small model that is shaky at maths word problems, and fine-tune it on reasoning traces written by a much bigger reasoning model. This is **distillation**: the small model copies the big model's step-by-step thinking, one token at a time. No rewards, no groups, just "here is a good answer, learn to write like this".

**How to open it in Colab:** go to [colab.research.google.com](https://colab.research.google.com), choose *File → Upload notebook* and pick this file. Then choose *Runtime → Change runtime type → T4 GPU*.

**Expected runtime on a free T4 (estimate):** about 45–60 minutes in total: roughly 10 minutes for each before/after check and 20–30 minutes of training.

**What you need:** a free Google account. A free Hugging Face account only if you want to upload the result at the end.

## 1. Check the GPU

You should see a Tesla T4 (or better). If you see an error, change the runtime type as described above.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install pinned libraries

We pin exact versions so this notebook keeps working when the libraries change. It was written for **TRL 1.16.0**. Colab already has PyTorch.

In [ ]:
!pip install -q trl==1.16.0 transformers==5.20.2 peft==0.21.2 datasets==5.1.0 accelerate==1.15.0

## 3. Settings

- **Student:** `Qwen/Qwen2.5-0.5B-Instruct`, about half a billion parameters.
- **Teacher traces:** [`camel-ai/gsm8k_distilled`](https://huggingface.co/datasets/camel-ai/gsm8k_distilled) (MIT license): about 7,500 GSM8K *training* problems, each with a long `<think>` trace and a final `\boxed{}` answer, produced with CAMEL-AI's DeepSeek-R1 distillation pipeline.
- **Test:** the first 200 problems of the GSM8K *test* split, which the traces never saw.

The system prompt asks for the same `<think>…</think><answer>…</answer>` format you used for GRPO.

In [ ]:
import re, statistics
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
TRACES = "camel-ai/gsm8k_distilled"
N_TRAIN = 1000      # traces to learn from
MAX_LENGTH = 1024   # tokens; longer traces are skipped so they fit a T4
N_EVAL = 200        # GSM8K test problems for the before/after check

SYSTEM_PROMPT = (
    "You are a careful problem solver. First think through the problem step by "
    "step inside <think> </think> tags. Then write only the final answer inside "
    "<answer> </answer> tags."
)
THINK_RE = re.compile(r"<think>(.*?)</think>", re.DOTALL)
BOXED_RE = re.compile(r"\\boxed\{([^{}]*)\}")
ANSWER_RE = re.compile(r"<answer>(.*?)</answer>", re.DOTALL)
NUMBER_RE = re.compile(r"-?\d+(?:\.\d+)?")

def to_number(text):
    """The last number in a piece of text (commas and $ removed), or None."""
    found = NUMBER_RE.findall((text or "").replace(",", "").replace("$", ""))
    return float(found[-1]) if found else None

def make_prompt(question):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}]

tokenizer = AutoTokenizer.from_pretrained(MODEL)
bf16 = torch.cuda.is_bf16_supported()   # False on a T4, so we train in fp16
print("bf16 supported:", bf16)

## 4. Look at one teacher trace

Always read your data before training on it. Notice how long the thinking is, and how it double-checks itself. That habit is exactly what we want the student to copy.

In [ ]:
raw = load_dataset(TRACES, split="train")
print(raw)
row = raw[0]
print("PROBLEM:\n", row["problem"])
print("\nTEACHER (first 1200 characters):\n", row["reasoning_solution"][:1200])

## 5. Turn traces into training examples

Each trace becomes a **prompt** (system + question) and a **completion** (the teacher's thinking inside `<think>` tags, then the number inside `<answer>` tags). We drop traces whose answer failed the dataset's own check, and traces longer than 1,024 tokens, which would not fit comfortably in T4 memory.

In [ ]:
def to_example(row):
    """Rewrite one teacher trace into our <think>/<answer> chat format."""
    think = THINK_RE.search(row["reasoning_solution"])
    boxed = BOXED_RE.findall(row["reasoning_solution"])
    answer = to_number(boxed[-1]) if boxed else None
    ok = bool(row["boxed_answer_success"]) and think is not None and answer is not None
    text = ""
    if ok:
        shown = int(answer) if answer == int(answer) else answer  # 18.0 -> 18
        text = f"<think>\n{think.group(1).strip()}\n</think>\n<answer>{shown}</answer>"
    return {"prompt": make_prompt(row["problem"]),
            "completion": [{"role": "assistant", "content": text}],
            "keep": ok}

def fits(ex):
    text = tokenizer.apply_chat_template(ex["prompt"] + ex["completion"], tokenize=False)
    return len(tokenizer(text)["input_ids"]) <= MAX_LENGTH

data = raw.shuffle(seed=0).map(to_example, remove_columns=raw.column_names)
data = data.filter(lambda ex: ex["keep"]).filter(fits)
print(f"{len(data)} traces fit in {MAX_LENGTH} tokens; using {min(N_TRAIN, len(data))}")
train_set = data.select(range(min(N_TRAIN, len(data)))).remove_columns("keep")
print(train_set[0]["completion"][0]["content"][:600])

## 6. A fair test

We grade on GSM8K test problems with **greedy** decoding (always pick the most likely next token), so the before and after runs are directly comparable. We record three things:

| measure | meaning |
|---|---|
| accuracy | share of problems where the final number is right |
| format | share of replies that close `</think>` and give an `<answer>` |
| tokens | average reply length, the "cost" of thinking |

To be fair to the untrained model, we accept the number from `<answer>`, then `\boxed{}`, then the last number in the reply.

In [ ]:
def predicted_number(text):
    """Prefer <answer>, then \\boxed{}, then the last number anywhere."""
    for pattern in (ANSWER_RE, BOXED_RE):
        found = pattern.findall(text)
        if found and to_number(found[-1]) is not None:
            return to_number(found[-1])
    return to_number(text)

@torch.no_grad()
def evaluate(model, problems, golds, max_new_tokens=1024, batch_size=16):
    model.eval()
    tokenizer.padding_side = "left"
    correct, formatted, lengths, samples = [], [], [], []
    for i in range(0, len(problems), batch_size):
        texts = [tokenizer.apply_chat_template(make_prompt(q), tokenize=False,
                                               add_generation_prompt=True)
                 for q in problems[i:i + batch_size]]
        inputs = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        for seq, gold in zip(out[:, inputs["input_ids"].shape[1]:], golds[i:i + batch_size]):
            reply = tokenizer.decode(seq, skip_special_tokens=True)
            lengths.append(int((seq != tokenizer.pad_token_id).sum()))
            correct.append(predicted_number(reply) == gold)
            formatted.append("</think>" in reply and ANSWER_RE.search(reply) is not None)
            samples.append(reply)
    return {"accuracy": statistics.fmean(correct), "format": statistics.fmean(formatted),
            "avg_tokens": statistics.fmean(lengths), "samples": samples}

test = load_dataset("openai/gsm8k", "main", split="test").select(range(N_EVAL))
problems = test["question"]
golds = [to_number(a.split("####")[-1]) for a in test["answer"]]

## 7. Before: the student on its own (about 5–10 minutes)

Write these numbers down. They are your baseline.

In [ ]:
student = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).to("cuda")
before = evaluate(student, problems, golds)
print({k: v for k, v in before.items() if k != "samples"})
print("\nFirst reply:\n", before["samples"][0][:800])
del student
torch.cuda.empty_cache()

## 8. Train (about 20–30 minutes)

This is ordinary supervised fine-tuning: predict the teacher's next token, over and over. Three choices matter:

- **LoRA** trains small add-on matrices instead of all the weights, so it fits a T4.
- **`completion_only_loss=True`**: the model is graded only on the teacher's answer, not on repeating the question.
- **fp16** instead of bf16, because a T4 has no bf16 support.

Watch the `loss` column go down, and `mean_token_accuracy` go up.

In [ ]:
tokenizer.padding_side = "right"
config = SFTConfig(
    output_dir="qwen-0.5b-distilled",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # 16 traces per optimizer step
    learning_rate=2e-4,              # LoRA likes a higher rate than full fine-tuning
    lr_scheduler_type="cosine",
    warmup_steps=5,
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    gradient_checkpointing=True,
    bf16=bf16,
    fp16=not bf16,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    seed=0,
)
peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                         target_modules="all-linear", task_type="CAUSAL_LM")
trainer = SFTTrainer(model=MODEL, args=config, train_dataset=train_set,
                     processing_class=tokenizer, peft_config=peft_config)
trainer.train()

## 9. After: same problems, same settings (about 10 minutes)

In [ ]:
after = evaluate(trainer.model, problems, golds)
print(f"{'':8}{'accuracy':>10}{'format':>9}{'tokens':>8}")
for name, r in (("before", before), ("after", after)):
    print(f"{name:8}{r['accuracy']:>10.1%}{r['format']:>9.1%}{r['avg_tokens']:>8.0f}")

## 10. Read the answers, not just the numbers

Look at a few replies side by side. Questions to ask yourself:

- Does the student now "think out loud", double-check, say "wait"? That is the teacher's style showing through.
- Did the replies get longer? By how much? Longer thinking costs time and money every time the model is used.
- Find one problem the student now gets wrong that it used to get right. Why might that be?

In [ ]:
for i in range(3):
    print("=" * 80)
    print("QUESTION:", problems[i])
    print("GOLD:", golds[i])
    print("\nBEFORE:\n", before["samples"][i][:700])
    print("\nAFTER:\n", after["samples"][i][:1200])

## 11. Save (and optionally share)

The adapter is only a few megabytes. To upload it, create a *write* token at huggingface.co/settings/tokens, run `login()` and set your own repo name.

In [ ]:
trainer.save_model("qwen-0.5b-distilled")
# from huggingface_hub import login
# login()
# trainer.model.push_to_hub("your-name/qwen-0.5b-distilled")
# tokenizer.push_to_hub("your-name/qwen-0.5b-distilled")

## What to try next

1. **Fewer traces.** Set `N_TRAIN = 200` and retrain. How much accuracy do you keep? (The s1 and LIMO papers report strong results from about 1,000 and about 800 carefully chosen examples, on much bigger students.)
2. **Shorter traces.** Set `MAX_LENGTH = 512`. The student sees only the teacher's shortest thinking. Does it think shorter too, and what happens to accuracy?
3. **Distill, then RL.** Use this adapter as the starting point for the GRPO run from module 5. A student that already "thinks" gives GRPO more right answers to learn from.